# MAREX 2027 complete flow

Precompute environments, run simulations, and generate every ICC 2027 figure. The flow `icc-2027-paper` produces everything the paper uses: the example runs with their illustrations, and the replications with their figures.

In [ ]:
import pathlib
import sys
sys.path.append("../..")
from exp_run_config import Config
from papers.y2027_marex.marex_graphics import build_figure_previews
from wbf_flow import build_marex_2027_flow_entries, display_flow_report, run_flow, setup_flow
Config.PROJECTNAME = "WaterBerryFarms"

In [ ]:
creation_style = "exist-ok"
# creation_style = "version"
# creation_style = "discard-old"
expruns_path = None
results_path = None
experiment = "marex2027-flow"
# the paper: the example runs with their illustrations, and the replications with their figures
run = "icc-2027-paper"
# run = "icc-2027-all"
# run = "icc-2027-replicated"
# the workspace of the flow, ~/WORK/WBF-Flows/<flow_name>
flow_name = "icc-2027-paper"
# reuse the completed runs and replications with the same configuration (e.g. to continue a partial flow)
skip_completed = False
# replicated collections: run only the first N replications of every base run (a partial flow, spread over
# the maps); None runs all of them. Continue a partial flow with a larger N and skip_completed = True.
replications = None

In [ ]:
if expruns_path is not None:
    Config().set_exprun_path(pathlib.Path(expruns_path))
if results_path is not None:
    Config().set_results_path(pathlib.Path(results_path))
flow_expruns, flow_results, notebooks_path = setup_flow(
    flow_name,
    ["environment", "policy", "estimator", "score",
     "marex2027-run", "marex2027-aggregate", "marex2027-figure", "marex2027-flow"])
entries = build_marex_2027_flow_entries(
    experiment, run, creation_style, skip_completed=skip_completed,
    replications=replications)
[(entry["name"], entry["notebook"]) for entry in entries]

In [ ]:
flow_error = None
try:
    run_flow(
        entries, flow_expruns, flow_results, notebooks_path)
except Exception as error:
    flow_error = error

In [ ]:
final_results_path = pathlib.Path(
    flow_results, "marex2027-figure")
flow_exp = Config().get_experiment(
    experiment, run, create_data_dir=False)
preview_files = build_figure_previews(
    flow_exp["figure-experiment"],
    flow_exp["figures"] + (flow_exp.get("aggregate-figures") or []))
report = display_flow_report(
    entries, flow_results, notebooks_path, final_results_path,
    flow_error=flow_error, preview_files=preview_files)
if flow_error is not None:
    raise flow_error
if not report["all-results-present"]:
    raise Exception("Flow finished without all expected stage results.")